# Embedding & Reranking — IO spec for llama.cpp (`llama-server`)

There are **three modes**, each a different server flag + endpoint combo:

| # | Server flag | Endpoint | Input | Output |
|---|---|---|---|---|
| A | `--embedding` | `POST /embedding` | list of texts | one float vector per text |
| B | `--reranking` | `POST /reranking` | query + 1 doc | relevance scalar |
| C | `--reranking` | `POST /reranking` | query + N docs | N relevance scalars (ranked) |

**Score types:**
- **Cosine similarity** (Mode A): query and doc are encoded *independently* → two vectors → you compute cosine. Works because vectors are L2-normalised (‖v‖=1), so cosine = dot product.
- **Relevance scalar** (Modes B/C): query and doc are fed *together* into a cross-encoder that outputs a single number directly. Not cosine. Not bounded. Not comparable across different reranker models.

In [1]:
import math, subprocess, time, requests
from pathlib import Path
from shutil import which

PORT = 8091
BASE = f"http://127.0.0.1:{PORT}"
M    = Path("/Users/e/.models/embed")

In [39]:
def start(model, *flags):
    # kills any running llama-server, boots a new one, waits for /health
    subprocess.run(["pkill", "-9", "-f", "llama-server"], capture_output=True)
    time.sleep(0.5)
    p = subprocess.Popen(
        [which("llama-server"), "-m", model, "--port", str(PORT), *flags],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) # surpress verbose logs
    for _ in range(60):
        try:
            if requests.get(f"{BASE}/health", timeout=2).ok: return p
        except: pass
        time.sleep(1)
    p.kill(); raise TimeoutError("server not ready")


def vec(item):
    e = item["embedding"]
    return e[0] if isinstance(e[0], list) else e  # unwrap ColBERT outer list

def cosine(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x*x for x in a)) * math.sqrt(sum(x*x for x in b)))

In [50]:
models = sorted(M.glob("*.gguf"))
#models = models1[0:1] # only the first model
models = models[1:2] + models[-2:-1] # pick only the second model and the last two models
models

[PosixPath('/Users/e/.models/embed/bge-reranker-v2-m3-q8_0.gguf'),
 PosixPath('/Users/e/.models/embed/qwen3-reranker-0.6b-q8_0.gguf')]

In [51]:
# ── inputs ────────────────────────────────────────────────────────
TEXTS = [                          # input → --embedding /embedding  (list of strings to encode)
    "The cat sat on the mat.",
    "Dogs are loyal companions.",
]
QUERY = "professional marathon racing shoe"# input → --reranking /reranking  (the query side)
DOCS  = [                          # input → --reranking /reranking  (documents to rank)
    "Nike Air Zoom: lightweight shoe, good for long runs",#"Stock market crash 2008.",
    "Adidas Adizero: worn by world record holders, sub-2hr marathon performance",#"Flip flops: casual beach sandals", #"A kitten on a rug.",
    "Asics Metaspeed: elite carbon-plate racing flat, podium finishes", #"Garden clogs: rubber slip-on shoes",#"Quantum mechanics.",
]

# ── (flag, endpoint, body, expected output shape) ─────────────────
COMBOS = [
    ("--embedding", 
     "/embedding",
     {"content": TEXTS},
     "list[ {index: int, embedding: [[float, ...]]} ]"),

    ("--reranking", 
     "/reranking",
     {"query": QUERY, "documents": DOCS},
     "{ results: list[ {index: int, relevance_score: float} ] }"),
]

In [52]:
def truncate_vecs(obj):
    """Recursively replace any list-of-numbers with '<N dimensions>'."""
    if isinstance(obj, list):
        if obj and isinstance(obj[0], (int, float)) and not isinstance(obj[0], bool):
            return f"<{len(obj)} dimensions>"
        return [truncate_vecs(v) for v in obj]
    if isinstance(obj, dict):
        return {k: truncate_vecs(v) for k, v in obj.items()}
    return obj

# ── model × combo matrix ──────────────────────────────────────────
# not every model supports every flag; failures are caught and recorded
results = {}

for m in models:
    results[m.name] = {}
    for flag, endpoint, body, expected_out in COMBOS:
        key = f"{flag} → {endpoint}"
        try:
            p = start(m, flag)
            raw = requests.post(f"{BASE}{endpoint}", json=body)
            p.terminate()
            results[m.name][key] = {"ok": True, "input": body.values(), "got": truncate_vecs(raw.json())} #"expected": expected_out, "got": truncate_vecs(raw.json())}
            
        except Exception as e:
            results[m.name][key] = {"ok": False, "error": str(e)}

results

{'bge-reranker-v2-m3-q8_0.gguf': {'--embedding → /embedding': {'ok': True,
   'input': dict_values([['The cat sat on the mat.', 'Dogs are loyal companions.']]),
   'got': [{'index': 0,
     'embedding': ['<1024 dimensions>',
      '<1024 dimensions>',
      '<1024 dimensions>',
      '<1024 dimensions>',
      '<1024 dimensions>',
      '<1024 dimensions>',
      '<1024 dimensions>',
      '<1024 dimensions>',
      '<1024 dimensions>']},
    {'index': 1,
     'embedding': ['<1024 dimensions>',
      '<1024 dimensions>',
      '<1024 dimensions>',
      '<1024 dimensions>',
      '<1024 dimensions>',
      '<1024 dimensions>',
      '<1024 dimensions>',
      '<1024 dimensions>',
      '<1024 dimensions>']}]},
  '--reranking → /reranking': {'ok': True,
   'input': dict_values(['professional marathon racing shoe', ['Nike Air Zoom: lightweight shoe, good for long runs', 'Adidas Adizero: worn by world record holders, sub-2hr marathon performance', 'Asics Metaspeed: elite carbon-plate raci

In [57]:

# ── Pointwise cross-encoding: one HTTP request per (query, doc) pair ─────────
# The /reranking endpoint accepts a batch, but each pair is scored independently.
# Here we call it one-at-a-time to make that explicit, then show the aggregated
# result looks identical to the single batched call.

RERANKER = next(m for m in sorted(M.glob("*.gguf")) if "bge-reranker-v2" in m.name)
print(f"Model: {RERANKER.name}\n")
p = start(RERANKER, "--reranking")

QUERY2 = "professional marathon racing shoe"
DOCS2  = [
    "Nike Air Zoom: lightweight shoe, good for long runs",
    "Adidas Adizero: worn by world record holders, sub-2hr marathon performance",
    "Asics Metaspeed: elite carbon-plate racing flat, podium finishes",
]

print("── Isolated (1 doc per request) ──────────────────────────────")
isolated = {}
for i, doc in enumerate(DOCS2):
    r = requests.post(f"{BASE}/reranking", json={"query": QUERY2, "documents": [doc]})
    result = r.json()
    score  = result["results"][0]["relevance_score"]
    tokens = result["usage"]["prompt_tokens"]
    isolated[i] = score
    print(f"  Request {i+1}: query + [{i}]  →  {tokens:3d} tokens  score={score:.6f}")

print()
print("── Batched (all docs in one request) ─────────────────────────")
r = requests.post(f"{BASE}/reranking", json={"query": QUERY2, "documents": DOCS2})
result  = r.json()
batched = {res["index"]: res["relevance_score"] for res in result["results"]}
tokens  = result["usage"]["prompt_tokens"]
for i, doc in enumerate(DOCS2):
    print(f"  [{i}]  score={batched[i]:.6f}")
print(f"  total tokens reported: {tokens}")

print()
print("── Delta (isolated vs batched) ────────────────────────────────")
for i in range(len(DOCS2)):
    delta = abs(isolated[i] - batched[i])
    print(f"  [{i}]  Δ={delta:.2e}  {'≈ FP noise ✓' if delta < 1e-4 else 'LARGE ⚠️'}")

p.terminate()


Model: bge-reranker-v2-m3-q8_0.gguf

── Isolated (1 doc per request) ──────────────────────────────
  Request 1: query + [0]  →   23 tokens  score=-1.320991
  Request 2: query + [1]  →   28 tokens  score=0.617653
  Request 3: query + [2]  →   26 tokens  score=-1.892139

── Batched (all docs in one request) ─────────────────────────
  [0]  score=-1.313899
  [1]  score=0.619518
  [2]  score=-1.892139
  total tokens reported: 77

── Delta (isolated vs batched) ────────────────────────────────
  [0]  Δ=7.09e-03  LARGE ⚠️
  [1]  Δ=1.86e-03  LARGE ⚠️
  [2]  Δ=0.00e+00  ≈ FP noise ✓


In [ ]:
# ── Listwise reranking via LLM + GBNF grammar ────────────────────────────────
# Uses a chat LLM (not a reranker) via /v1/chat/completions.
# The grammar at listwise_ranking.gbnf forces output to {"ranking": [i, j, k]}
# so the model ranks all docs in one pass — each doc attends to all others.

import json

CHAT_MODEL   = Path("/Users/e/.models/OpenGVLab_InternVL3_5-1B-Q4_K_M.gguf")
GRAMMAR_PATH = Path("/Users/e/.models/listwise_ranking.gbnf")
GRAMMAR      = GRAMMAR_PATH.read_text()

QUERY3 = "professional marathon racing shoe"
DOCS3  = [
    "Nike Air Zoom: lightweight shoe, good for long runs",
    "Adidas Adizero: worn by world record holders, sub-2hr marathon performance",
    "Asics Metaspeed: elite carbon-plate racing flat, podium finishes",
    "Flip flops: casual beach sandals",
    "Garden clogs: rubber slip-on shoes",
]

doc_block = "\n".join(f"[{i}] {d}" for i, d in enumerate(DOCS3))
prompt = f"""Rank the following documents by relevance to the query. Most relevant first.
Respond ONLY with a JSON object: {{"ranking": [list of 0-based indices]}}

Query: {QUERY3}

Documents:
{doc_block}"""

p = start(CHAT_MODEL)   # plain --chat mode (default)

r = requests.post(f"{BASE}/v1/chat/completions", json={
    "messages": [{"role": "user", "content": prompt}],
    "grammar":  GRAMMAR,
    "temperature": 0,
})
raw_content = r.json()["choices"][0]["message"]["content"]
ranking     = json.loads(raw_content)["ranking"]

p.terminate()

print(f"Query: {QUERY3}\n")
print("Listwise ranking (LLM, sees all docs at once):")
for rank, idx in enumerate(ranking):
    label = DOCS3[idx] if idx < len(DOCS3) else f"[invalid index {idx}]"
    print(f"  #{rank+1}  [{idx}] {label}")
print(f"\nRaw output: {raw_content}")


Query: professional marathon racing shoe

Listwise ranking (LLM, sees all docs at once):
  #1  [1] Adidas Adizero: worn by world record holders, sub-2hr marathon performance
  #2  [2] Asics Metaspeed: elite carbon-plate racing flat, podium finishes
  #3  [0] Nike Air Zoom: lightweight shoe, good for long runs
  #4  [3] Flip flops: casual beach sandals
  #5  [4] Garden clogs: rubber slip-on shoes

Raw output: {"ranking": [1, 2, 0, 3, 4]}


In [64]:

# ── Dummy Substitution Test: proof of pointwise vs listwise ────────────────
# If the model is truly pointwise, doc0's score should be IDENTICAL regardless
# of what other docs are in the batch (since doc0 never "sees" them).
# If listwise, doc0's score changes based on context.
#
# NOTE: /reranking returns results sorted by score (highest first), NOT by
# original index. Must look up by index, not by position in the list.

def score_of(response, target_index):
    """Get the relevance score for a specific document index (not rank position)."""
    return next(r["relevance_score"] for r in response["results"] if r["index"] == target_index)

RERANKER_DUMMY = next(m for m in sorted(M.glob("*.gguf")) if "bge-reranker-v2" in m.name)
print(f"Model: {RERANKER_DUMMY.name}\n")
p = start(RERANKER_DUMMY, "--reranking")

QUERY_DUMMY = "professional marathon racing shoe"
DOC0 = "Nike Air Zoom: lightweight shoe, good for long runs"

# ── Scenario A: original context ──────────────────────────────────
DOC1_A = "Adidas Adizero: worn by world record holders, sub-2hr marathon performance"
DOC2_A = "Asics Metaspeed: elite carbon-plate racing flat, podium finishes"
r_a = requests.post(f"{BASE}/reranking", json={"query": QUERY_DUMMY, "documents": [DOC0, DOC1_A, DOC2_A]})
score_a = score_of(r_a.json(), 0)

# ── Scenario B: same doc0, completely different context ──────────────────────
DOC1_B = "A red brick wall"
DOC2_B = "Quantum entanglement in physics"
r_b = requests.post(f"{BASE}/reranking", json={"query": QUERY_DUMMY, "documents": [DOC0, DOC1_B, DOC2_B]})
score_b = score_of(r_b.json(), 0)

# ── Scenario C: same doc0, different fillers again ──────────────────────────
DOC1_C = "Ice cream comes in many flavors"
DOC2_C = "The number 42 is important in sci-fi"
r_c = requests.post(f"{BASE}/reranking", json={"query": QUERY_DUMMY, "documents": [DOC0, DOC1_C, DOC2_C]})
score_c = score_of(r_c.json(), 0)

p.terminate()

print("Dummy Substitution Test – is doc0 score stable?")
print(f"  Scenario A (Adidas + Asics):     {score_a:.8f}")
print(f"  Scenario B (red brick + QM):     {score_b:.8f}")
print(f"  Scenario C (ice cream + 42):     {score_c:.8f}")
print()

max_delta = max(abs(score_a - score_b), abs(score_a - score_c), abs(score_b - score_c))
print(f"Max delta across scenarios: {max_delta:.2e}")
print()

if max_delta < 1e-3:
    print("✓ POINTWISE CONFIRMED: doc0 score is stable across different batch contexts")
else:
    print("⚠️  LISTWISE SUSPECTED: doc0 score changed significantly based on batch context")


Model: bge-reranker-v2-m3-q8_0.gguf

Dummy Substitution Test – is doc0 score stable?
  Scenario A (Adidas + Asics):     -1.31389880
  Scenario B (red brick + QM):     -1.31746757
  Scenario C (ice cream + 42):     -1.31910264

Max delta across scenarios: 5.20e-03

⚠️  LISTWISE SUSPECTED: doc0 score changed significantly based on batch context


In [65]:

# ── Competitive Substitution Test ─────────────────────────────────────────────
# Stronger proof of pointwise behaviour.
# We put doc0 in two extreme contexts:
#
#   Scenario HARD  – doc0 is the *worst* doc in the batch
#                    (companions are all clearly better marathon shoes)
#   Scenario EASY  – doc0 is the *best* doc in the batch
#                    (companions are nonsense with zero shoe relevance)
#
# Listwise prediction : HARD score << EASY score  (big context-driven swing)
# Pointwise prediction: HARD score ≈ EASY score   (absolute, context-blind)
#
# We also score doc0 *alone* as a reference.

RERANKER2 = next(m for m in sorted(M.glob("*.gguf")) if "bge-reranker-v2" in m.name)
print(f"Model: {RERANKER2.name}\n")
p = start(RERANKER2, "--reranking")

QUERY_C  = "professional marathon racing shoe"
DOC0_C   = "Nike Air Zoom: lightweight shoe, good for long runs"

# companions that are CLEARLY SUPERIOR to doc0 on the query
HARD_COMPANIONS = [
    "Adidas Adizero Prime X: wore by Eliud Kipchoge in the sub-2hr marathon attempt",
    "Asics Metaspeed Sky+: set world half-marathon record, elite carbon-plate racer",
    "Nike Vaporfly NEXT%3: official world-record marathon shoe, 4% energy return",
    "New Balance SC Elite v4: 2024 Olympic marathon podium, lightest racing flat",
    "Saucony Endorphin Pro 4: sub-elite marathon racer, 45% Peba foam carbon plate",
]

# companions that are CLEARLY INFERIOR to doc0 on the query (random noise)
EASY_COMPANIONS = [
    "A banana is a yellow fruit that grows in tropical climates",
    "The Eiffel Tower stands 330 metres tall in the centre of Paris",
    "Photosynthesis converts sunlight into glucose inside plant cells",
    "The boiling point of water is 100 degrees Celsius at sea level",
    "Jupiter is the largest planet in the Solar System",
]

# ── score doc0 alone (isolated reference) ─────────────────────────
r_solo  = requests.post(f"{BASE}/reranking",
                        json={"query": QUERY_C, "documents": [DOC0_C]})
score_solo = r_solo.json()["results"][0]["relevance_score"]

# ── score doc0 among champions ─────────────────────────────────────
r_hard = requests.post(f"{BASE}/reranking",
                       json={"query": QUERY_C,
                             "documents": [DOC0_C] + HARD_COMPANIONS})
score_hard = score_of(r_hard.json(), 0)   # score_of defined in cell 9

# ── score doc0 among nonsense ──────────────────────────────────────
r_easy = requests.post(f"{BASE}/reranking",
                       json={"query": QUERY_C,
                             "documents": [DOC0_C] + EASY_COMPANIONS})
score_easy = score_of(r_easy.json(), 0)

p.terminate()

print(f"Query : {QUERY_C}")
print(f"Doc0  : {DOC0_C}\n")
print(f"  Isolated (alone)                    : {score_solo:.8f}  ← reference")
print(f"  HARD context (surrounded by champs) : {score_hard:.8f}")
print(f"  EASY context (surrounded by noise)  : {score_easy:.8f}")
print()

delta_hard_easy = abs(score_hard - score_easy)
delta_vs_solo   = max(abs(score_hard - score_solo), abs(score_easy - score_solo))

print(f"  HARD vs EASY delta : {delta_hard_easy:.2e}")
print(f"  Max delta vs solo  : {delta_vs_solo:.2e}")
print()

THRESHOLD = 0.05   # a genuinely listwise effect would be >> 0.05
if delta_hard_easy < THRESHOLD:
    print(f"✓ POINTWISE CONFIRMED  (delta {delta_hard_easy:.2e} < {THRESHOLD})")
    print("  doc0 score is insensitive to batch context → cross-encoder / pointwise")
else:
    print(f"⚠️  LISTWISE SUSPECTED  (delta {delta_hard_easy:.2e} ≥ {THRESHOLD})")
    print("  doc0 score changed based on what else is in the batch → context-dependent")


Model: bge-reranker-v2-m3-q8_0.gguf

Query : professional marathon racing shoe
Doc0  : Nike Air Zoom: lightweight shoe, good for long runs

  Isolated (alone)                    : -1.32099068  ← reference
  HARD context (surrounded by champs) : -1.31886446
  EASY context (surrounded by noise)  : -1.31579137

  HARD vs EASY delta : 3.07e-03
  Max delta vs solo  : 5.20e-03

✓ POINTWISE CONFIRMED  (delta 3.07e-03 < 0.05)
  doc0 score is insensitive to batch context → cross-encoder / pointwise
